# Blinkit Data Analytics — Payment analysis
DuckDB SQL + Python | Yash Prajapati

## 1. Setup

### 1.1 Libraries

In [1]:
import warnings, math, textwrap
warnings.filterwarnings('ignore')
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
print('duckdb', duckdb.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

duckdb 1.5.5 | pandas 3.0.2 | numpy 2.4.4


### 1.2 Load cleaned workbook

In [2]:
XLSX = 'data/Blinkit_analysis_new.xlsx'
book = pd.read_excel(XLSX, sheet_name=None)
geo = pd.read_csv('data/city_state_zone.csv')
for name, df in book.items():
    print(f'{name:28s} {df.shape[0]:>6,} rows  {df.shape[1]:>3} cols')

Data_Quality_Report              46 rows    3 cols
Orders_Customer_Info          5,000 rows   16 cols
Orders_Raw_Archive            5,000 rows   23 cols
Order_Line_Items              5,000 rows   14 cols
Delivery_Performance          5,000 rows    8 cols
Customer_Feedback             5,000 rows    8 cols
Customers                     2,500 rows   11 cols
Products                        268 rows   10 cols
Inventory_Analysis              268 rows   18 cols
Data_Dictionary                  44 rows    4 cols
Reference_Parameters             11 rows    5 cols
Relational_Diagram                0 rows    0 cols
Pivot_Payment_Method              7 rows    7 cols
Pivot_Customer_Segment            7 rows    5 cols
Pivot_Category_Sales             14 rows    4 cols
Pivot_Monthly_Trend              24 rows    6 cols
Pivot_Area_Delivery              23 rows    6 cols
Pivot_Inventory_Movement          6 rows    7 cols
Pivot_Category_Stock             14 rows    8 cols


### 1.3 Create DuckDB database and load tables

In [3]:
con = duckdb.connect('blinkit.duckdb')
load = {'orders_src':'Orders_Raw_Archive','items_src':'Order_Line_Items','delivery_src':'Delivery_Performance',
        'feedback_src':'Customer_Feedback','customers_src':'Customers','products_src':'Products','inventory_src':'Inventory_Analysis'}
for tbl, sheet in load.items():
    df = book[sheet].copy()
    df.columns = [c.strip() for c in df.columns]
    con.register('tmp_df', df)
    con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df')
con.register('geo_df', geo)
con.execute('CREATE OR REPLACE TABLE geo AS SELECT * FROM geo_df')
con.execute("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name").df()

,table_name,estimated_size
0,customers_src,2500
1,delivery_src,5000
2,feedback_src,5000
3,geo,316
4,inventory_src,268
5,items_src,5000
6,orders_src,5000
7,products_src,268


### 1.4 Typed analysis views

In [4]:
con.execute('''
CREATE OR REPLACE VIEW orders AS
SELECT order_id, customer_id,
       strptime(order_date, '%d-%m-%Y %H:%M')              AS order_ts,
       CAST(strptime(order_date, '%d-%m-%Y %H:%M') AS DATE) AS order_date,
       strptime(promised_delivery_time, '%d-%m-%Y %H:%M')   AS promised_ts,
       strptime(actual_delivery_time,  '%d-%m-%Y %H:%M')    AS actual_ts,
       delivery_status, order_total, payment_method, delivery_partner_id, store_id,
       CAST(delivery_time_minutes AS INTEGER)               AS delay_min,
       distance_km, reasons_if_delayed, customer_name,
       trim(area) AS area, pincode, customer_segment,
       CAST(registration_date AS DATE)                      AS registration_date,
       order_day_of_week, order_time_slot, order_value_segment,
       CASE WHEN is_weekend = 'Yes' THEN 1 ELSE 0 END       AS is_weekend
FROM orders_src ''')

con.execute('''
CREATE OR REPLACE VIEW items AS
SELECT order_id, product_id, quantity, unit_price, product_name, category, brand,
       price, mrp, margin_percentage, shelf_life_days, min_stock_level, max_stock_level, line_total,
       line_total * margin_percentage / 100.0 AS margin_value
FROM items_src ''')

con.execute('''
CREATE OR REPLACE VIEW feedback AS
SELECT feedback_id, order_id, customer_id, rating, feedback_category, sentiment,
       CAST(feedback_date AS DATE) AS feedback_date
FROM feedback_src ''')

con.execute('''
CREATE OR REPLACE VIEW f_sales AS
SELECT o.order_id, o.customer_id, o.order_ts, o.order_date,
       date_trunc('month', o.order_date)  AS order_month,
       extract(hour FROM o.order_ts)      AS order_hour,
       o.order_day_of_week, o.is_weekend, o.order_time_slot, o.order_value_segment,
       o.payment_method, o.customer_segment, o.registration_date, o.customer_name,
       o.area, g.state, g.zone, g.city_tier,
       i.product_id, i.product_name, i.category, i.brand,
       i.quantity, i.line_total AS revenue, i.margin_value, i.margin_percentage,
       i.price, i.mrp, i.shelf_life_days,
       o.delay_min, o.distance_km, o.delivery_status,
       CASE WHEN o.delivery_status = 'On Time' THEN 1 ELSE 0 END AS is_on_time_status,
       CASE WHEN o.delay_min > 0 THEN 1 ELSE 0 END               AS is_late_minutes,
       f.rating, f.sentiment, f.feedback_category
FROM orders o
JOIN items i    ON i.order_id = o.order_id
LEFT JOIN geo g ON g.area     = o.area
LEFT JOIN feedback f ON f.order_id = o.order_id ''')

con.execute('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_date) AS first_day, MAX(order_date) AS last_day FROM f_sales').df()

,fact_rows,orders,first_day,last_day
0,5000,5000,2023-03-16,2024-11-04


### 1.5 Query helper

In [5]:
def q(sql, con=con):
    return con.execute(textwrap.dedent(sql)).df()

def pct(x, n):
    return round(100.0 * x / n, 2) if n else 0.0

q('SELECT COUNT(*) AS rows_in_fact_view FROM f_sales')

,rows_in_fact_view
0,5000


## 10. Payment behaviour

### 10.1 Payment method performance

In [6]:
q('''
SELECT payment_method, COUNT(*) AS orders,
       round(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS order_share_pct,
       round(SUM(revenue), 0) AS revenue,
       round(100 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 2) AS revenue_share_pct,
       round(AVG(revenue), 0) AS avg_order_value,
       round(AVG(rating), 2) AS avg_rating,
       round(100.0 * COUNT(*) FILTER (WHERE is_on_time_status = 1) / COUNT(*), 2) AS on_time_pct
FROM f_sales GROUP BY 1 ORDER BY revenue DESC ''')

,payment_method,orders,order_share_pct,revenue,revenue_share_pct,avg_order_value,avg_rating,on_time_pct
0,Card,1285,25.70,"1,326,263.00",26.67,"1,032.00",3.36,69.96
1,Cash,1257,25.14,"1,231,327.00",24.76,980.00,3.33,69.13
2,UPI,1214,24.28,"1,221,420.00",24.56,"1,006.00",3.33,70.51
3,Wallet,1244,24.88,"1,193,405.00",24.00,959.00,3.36,68.01


### 10.2 Payment method by customer label

In [7]:
q('''
SELECT customer_segment AS label,
       COUNT(*) FILTER (WHERE payment_method = 'Cash')   AS cash,
       COUNT(*) FILTER (WHERE payment_method = 'Card')   AS card,
       COUNT(*) FILTER (WHERE payment_method = 'UPI')    AS upi,
       COUNT(*) FILTER (WHERE payment_method = 'Wallet') AS wallet,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Cash') / COUNT(*), 1) AS cash_share_pct
FROM f_sales GROUP BY 1 ORDER BY label ''')

,label,cash,card,upi,wallet,cash_share_pct
0,Inactive,305,300,284,301,25.60
1,New,305,324,302,291,25.00
2,Premium,318,315,290,345,25.10
3,Regular,329,346,338,307,24.90


### 10.3 Payment mix by zone

In [8]:
q('''
SELECT zone, COUNT(*) AS orders,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Cash') / COUNT(*), 1)   AS cash_pct,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'UPI') / COUNT(*), 1)    AS upi_pct,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Card') / COUNT(*), 1)   AS card_pct,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Wallet') / COUNT(*), 1) AS wallet_pct,
       round(AVG(revenue), 0) AS aov
FROM f_sales GROUP BY zone ORDER BY orders DESC ''')

,zone,orders,cash_pct,upi_pct,card_pct,wallet_pct,aov
0,South,1405,25.20,25.80,25.20,23.80,991.00
1,East,1042,26.50,23.90,25.10,24.50,"1,002.00"
2,Central,1009,23.70,24.00,25.60,26.80,984.00
3,West,695,24.60,22.20,28.50,24.70,987.00
4,North,617,26.30,24.00,25.80,24.00,"1,009.00"
5,North East,232,23.70,25.00,23.30,28.00,"1,010.00"


### 10.4 Payment method against order value band

In [9]:
pay_ct = q('''
SELECT payment_method, order_value_segment, COUNT(*) AS orders FROM f_sales GROUP BY 1, 2 ''')
tab = pay_ct.pivot(index='payment_method', columns='order_value_segment', values='orders').fillna(0)
chi2, p, dof, _ = stats.chi2_contingency(tab)
print(tab)
pd.DataFrame([{'test':'Chi-square: payment method vs order value band','chi2':round(chi2,3),'dof':dof,'p_value':round(p,4),
               'reading':'independent' if p > 0.05 else 'related'}])

order_value_segment  High Value  Low Value  Medium Value
payment_method                                          
Card                        447        415           423
Cash                        434        404           419
UPI                         407        416           391
Wallet                      412        415           417


,test,chi2,dof,p_value,reading
0,Chi-square: payment method vs order value band,2.19,6,0.90,independent


### 10.5 Payment share over time

In [10]:
q('''
SELECT strftime(order_month, '%Y-%m') AS month,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'UPI') / COUNT(*), 1)  AS upi_pct,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Cash') / COUNT(*), 1) AS cash_pct,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Card') / COUNT(*), 1) AS card_pct,
       round(100.0 * COUNT(*) FILTER (WHERE payment_method = 'Wallet') / COUNT(*), 1) AS wallet_pct
FROM f_sales GROUP BY order_month ORDER BY order_month ''')

,month,upi_pct,cash_pct,card_pct,wallet_pct
0,2023-03,17.50,27.50,27.50,27.50
1,2023-04,20.60,29.40,23.10,26.90
2,2023-05,22.50,22.50,32.20,22.80
3,2023-06,22.00,24.60,25.00,28.40
4,2023-07,24.20,25.80,23.80,26.20
5,2023-08,29.80,26.00,22.10,22.10
6,2023-09,24.40,21.00,28.20,26.30
7,2023-10,26.40,23.60,24.00,26.00
8,2023-11,26.00,24.50,24.90,24.50
9,2023-12,27.20,25.00,25.00,22.80
